# Data Processing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs('figures', exist_ok=True)

df = pd.read_csv('credit_approval/credit_approval.csv')

In [ ]:
# Preprocessing
from sklearn.preprocessing import LabelEncoder

df_clean = df.dropna().reset_index(drop=True)
print(f"Rows after dropping missing: {len(df_clean)}  (dropped {len(df) - len(df_clean)})")

CAT_COLS  = ['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13']
CONT_COLS = ['A2', 'A3', 'A8', 'A11', 'A14', 'A15']
FEATURES  = CAT_COLS + CONT_COLS
TARGET    = 'A16'

df_enc = df_clean.copy()
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col])
    encoders[col] = le

df_enc[TARGET] = (df_enc[TARGET] == '+').astype(int)

X             = df_enc[FEATURES].values.astype(float)
y             = df_enc[TARGET].values
feature_names = FEATURES

print(f"X shape: {X.shape}, class balance: {y.mean():.1%} approved (+)")

**Note:**
1. We drop all rows that contain N.A., because
   1. it only constitutes a very small fraction of the dataset
   2. imputation is risky when we do not know what each feature means
2. LabelEncoder() might be problematic for multi-category features because it artificially introduces ordering. Fortunately, none of these features contribute significantly (see later).

# Model Training

In [ ]:
# Model Selection via K-Fold CV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import warnings; warnings.filterwarnings('ignore')

param_dist = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1),
    param_distributions=param_dist,
    n_iter=60,
    scoring='roc_auc',
    cv=cv,
    return_train_score=False,
    n_jobs=-1,
    random_state=42,
    refit=False,
)
search.fit(X, y)

results    = pd.DataFrame(search.cv_results_)
AUC_FLOOR  = 0.80
candidates = results[results['mean_test_score'] > AUC_FLOOR]
best_idx   = candidates['std_test_score'].idxmin()
best_params = results.loc[best_idx, 'params']

print(f"Candidates above AUC >= {AUC_FLOOR}: {len(candidates)}")
print(f"Selected mean AUC: {results.loc[best_idx, 'mean_test_score']:.4f}  ",
      f"std: {results.loc[best_idx, 'std_test_score']:.4f}")
print(f"Params: {best_params}")

best_rf = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
best_rf.fit(X, y)

**Note:**
1. We want a model that is (1) fairly accurate and (2) stable.
2. The goal is not to **maximize** OOS prediction accuracy, but to identify which features drive decisions, in what direction, and how they interact. Interpretability is only meaningful when our model is stable.
3. We use a RandomForest model and randomized search to tune hyperparameters that yield the most stable results across folds while maintaining a high AUC.

# Feature Analysis and Interpretability

## SHAP and Feature Importance

In [ ]:
# Per-Fold: Permutation Importance + SHAP Signs
from sklearn.inspection import permutation_importance
import shap

fold_perm_imp   = []
fold_shap_signs = []

for k, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    rf = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)

    perm = permutation_importance(
        rf, X_val, y_val, scoring='roc_auc', n_repeats=5, random_state=42, n_jobs=-1
    )
    fold_perm_imp.append(perm.importances_mean)

    expl = shap.TreeExplainer(rf)
    sv   = expl.shap_values(X_val)
    sv1  = sv[1] if isinstance(sv, list) else sv[..., 1]
    fold_shap_signs.append(sv1.mean(axis=0))

    print(f"  fold {k+1:2d}/10", end='\r')

fold_perm_imp   = np.array(fold_perm_imp)
fold_shap_signs = np.array(fold_shap_signs)
print('Per-fold analysis complete.')

In [ ]:
# SHAP on Final Model (all data)
explainer_final = shap.TreeExplainer(best_rf)
sv_final        = explainer_final.shap_values(X)
shap_approval   = sv_final[1] if isinstance(sv_final, list) else sv_final[..., 1]

mean_abs_shap = np.abs(shap_approval).mean(axis=0)
top_idx       = np.argsort(mean_abs_shap)[::-1]
top_features  = [feature_names[i] for i in top_idx]
TOP_N = 8

print('Feature ranking by mean |SHAP|:')
for rank, i in enumerate(top_idx):
    print(f"  {rank+1:2d}. {feature_names[i]:4s}  {mean_abs_shap[i]:.4f}")

In [ ]:
# Feature Importance: Mean +/- Std Across 10 Folds
mean_imp = fold_perm_imp.mean(axis=0)
std_imp  = fold_perm_imp.std(axis=0)
order    = np.argsort(mean_imp)[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(feature_names))
ax.bar(x, mean_imp[order], yerr=std_imp[order], capsize=4,
       color='steelblue', alpha=0.8, error_kw=dict(ecolor='black', lw=1.2))
ax.set_xticks(x)
ax.set_xticklabels([feature_names[i] for i in order], rotation=45, ha='right')
ax.set_ylabel('Permutation importance (AUC drop)')
ax.set_title('Feature Importance -- Mean +/- Std Across 10 Folds')
plt.tight_layout()
plt.savefig('figures/fig1_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Direction: Mean SHAP Sign Stability Across 10 Folds
mean_sign = fold_shap_signs.mean(axis=0)
std_sign  = fold_shap_signs.std(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ecc71' if s > 0 else '#e74c3c' for s in mean_sign[order]]
ax.bar(x, mean_sign[order], yerr=std_sign[order], capsize=4,
       color=colors, alpha=0.8, error_kw=dict(ecolor='black', lw=1.2))
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xticks(x)
ax.set_xticklabels([feature_names[i] for i in order], rotation=45, ha='right')
ax.set_ylabel('Mean SHAP value (approval direction)')
ax.set_title('Feature Direction -- Mean SHAP Across 10 Folds')
plt.tight_layout()
plt.savefig('figures/fig2_shap_direction.png', dpi=150, bbox_inches='tight')
plt.show()